In [ ]:
import json
import logging
import sys
import os
import importlib
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [ ]:
sys.path.append(os.path.abspath(".."))
import config
importlib.reload(config)

from config import (
    DATASET_CLEAN_DIR,
    FEATURE_COLUMNS,
    MODELS_DIR,
    TARGET_COLUMN
)
from training.temporal_cv import expanding_month_splits


In [ ]:
df = pd.read_csv(DATASET_CLEAN_DIR / "features_leptospirosis_monthly.csv")
df["month_start"] = pd.to_datetime(df["month_start"])

train_df = df[df["month_start"] < "2025-01-01"].copy()
test_df = df[df["month_start"] >= "2025-01-01"].copy()
X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df[TARGET_COLUMN]
y_train_log = np.log1p(y_train)
cv_splits = expanding_month_splits(
    train_df, n_splits=5, min_train_months=max(2, train_df["month_start"].nunique() // 2)
)

print(f"Total dataset samples: {len(df)} rows ({df['month_start'].min().strftime('%Y-%m-%d')} to {df['month_start'].max().strftime('%Y-%m-%d')})")

In [ ]:
# Validasi temporal hanya pada periode latih; seluruh kecamatan dipisah per bulan.
models_cv = {
    "RandomForest (d=4)": RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42, n_jobs=-1),
    "ExtraTrees (d=4)": ExtraTreesRegressor(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42),
    "XGBoost (d=2)": XGBRegressor(n_estimators=100, max_depth=2, learning_rate=0.03, random_state=42),
    "Ridge (a=5.0)": Ridge(alpha=5.0, random_state=42),
}

print("=== 5-FOLD CROSS VALIDATION FOR LEPTOSPIROSIS SUB-MODELS ===")
for name, m in models_cv.items():
    cv_mae, cv_rmse, cv_r2 = [], [], []
    for tr_idx, val_idx in cv_splits:
        X_tr, y_tr_k = X_train.iloc[tr_idx], y_train_log.iloc[tr_idx]
        X_val, y_val_k = X_train.iloc[val_idx], y_train.iloc[val_idx]
        m.fit(X_tr, y_tr_k)
        p_val = np.clip(np.expm1(m.predict(X_val)), 0, None)
        cv_mae.append(mean_absolute_error(y_val_k, p_val))
        cv_rmse.append(np.sqrt(mean_squared_error(y_val_k, p_val)))
        cv_r2.append(r2_score(y_val_k, p_val))
    print(f"{name:<20} -> MAE: {np.mean(cv_mae):.4f}, RMSE: {np.mean(cv_rmse):.4f}, R2 (CV): {np.mean(cv_r2):.4f} ({np.mean(cv_r2)*100:.2f}%)")

In [ ]:
# Evaluasi Ensemble Model dengan 5-Fold Cross-Validation
cv_mae, cv_rmse, cv_r2 = [], [], []
for tr_idx, val_idx in cv_splits:
    X_tr, y_tr_k = X_train.iloc[tr_idx], y_train_log.iloc[tr_idx]
    X_val, y_val_k = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    m_rf = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42, n_jobs=-1).fit(X_tr, y_tr_k)
    m_et = ExtraTreesRegressor(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42).fit(X_tr, y_tr_k)
    m_xgb = XGBRegressor(n_estimators=100, max_depth=2, learning_rate=0.03, random_state=42).fit(X_tr, y_tr_k)
    m_ridge = Ridge(alpha=5.0, random_state=42).fit(X_tr, y_tr_k)
    
    p_rf = np.expm1(m_rf.predict(X_val))
    p_et = np.expm1(m_et.predict(X_val))
    p_xgb = np.expm1(m_xgb.predict(X_val))
    p_ridge = np.expm1(m_ridge.predict(X_val))
    
    p_blend = np.clip(0.40 * p_rf + 0.35 * p_et + 0.15 * p_xgb + 0.10 * p_ridge, 0, None)
    
    cv_mae.append(mean_absolute_error(y_val_k, p_blend))
    cv_rmse.append(np.sqrt(mean_squared_error(y_val_k, p_blend)))
    cv_r2.append(r2_score(y_val_k, p_blend))

print("=== EVALUATION OF ENSEMBLE BLENDING MODEL (5-FOLD CV) ===")
print(f"MAE : {np.mean(cv_mae):.4f} cases/month")
print(f"RMSE: {np.mean(cv_rmse):.4f}")
print(f"R2  : {np.mean(cv_r2):.4f} ({np.mean(cv_r2)*100:.2f}%)")

In [ ]:
from training.ensemble import LeptospirosisEnsembleModel

model = LeptospirosisEnsembleModel()
model.fit(X_train, y_train_log)
y_pred_clipped = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_clipped)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_clipped))
r2 = r2_score(y_test, y_pred_clipped)

print("=== HOLDOUT 2025: FINAL LEPTOSPIROSIS ENSEMBLE ===")
print(f"MAE : {mae:.4f} cases/month")
print(f"RMSE: {rmse:.4f}")
print(f"R2  : {r2:.4f} ({r2*100:.2f}%)")

importances = model.feature_importances_
feat_importance_df = (
    pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": importances})
    .sort_values(by="importance", ascending=False)
    .reset_index(drop=True)
)
print("\nTop 5 Feature Importances:\n" + feat_importance_df.head(5).to_string(index=False))